In [ ]:
import os
import dotenv
import os, json, time

dotenv.load_dotenv()

GROQ_API_KEY = os.environ["GROQ_API_KEY"]
PAGEINDEX_API_KEY = os.environ["PAGEINDEX_API_KEY"]

# print(PAGEINDEX_API_KEY if PAGEINDEX_API_KEY else "No API key found")
# print(GROQ_API_KEY if GROQ_API_KEY else "No API key found")

In [ ]:

from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.7, streaming=True)

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [ ]:
from langchain.messages import SystemMessage, HumanMessage,AIMessage

messages=[
    SystemMessage("You are a poetry expert"),
    HumanMessage("Write a poem on artificial intelligence")
]

response=model.invoke(messages)
response.content

In [ ]:
for chunk in response:
    print(chunk)



In [ ]:
from pydantic import BaseModel,Field


In [ ]:
pdf = './hirok.pdf'
pdf

In [ ]:
from pageindex import PageIndexClient
 
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

In [ ]:
result = pi_client.submit_document(pdf)
doc_id = result["doc_id"]

In [ ]:
status = pi_client.get_document(doc_id)["status"]
if status == "completed":
    print('Document processing completed')

In [ ]:
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

In [ ]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

In [ ]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

In [ ]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

In [ ]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────
# Re-run this cell after any edit (kernel keeps the old function until you do).

from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

_tree_search_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    streaming=False,
)


def llm_tree_search(query: str, tree: list, llm=None) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    chat = llm 
    response = model.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()
    if content.startswith("```"):
        content = content.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(content)

In [ ]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "what is the first job of hirok?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree, model)

print("🧠 LLM Reasoning:")
# print(result.get("thinking", "N/A"))
for chunk in result.get("thinking", "N/A").split("."):
    print(chunk)
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))